In [1]:
# IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Modèle SARIMAX
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Affichage
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Imports effectués avec succès.")

Imports effectués avec succès.


In [57]:
# ============================================================
# 51. VARIABLES EXOGÈNES POUR SARIMAX
# ============================================================

variables_exogenes = [
    "heure_sin",
    "heure_cos",
    "jour_semaine_sin",
    "jour_semaine_cos",
    "mois_sin",
    "mois_cos",
    "weekend_utc",
    "ferie",
    "vacances"
]

X_sarimax = df_model[variables_exogenes].copy()

# Conversion des variables booléennes en 0/1
X_sarimax["weekend_utc"] = (
    X_sarimax["weekend_utc"].astype(int)
)

X_sarimax["ferie"] = (
    X_sarimax["ferie"].astype(int)
)

print("Variables exogènes :")
print(X_sarimax.columns.tolist())

print("\nDimensions :", X_sarimax.shape)

print("\nValeurs manquantes :")
print(X_sarimax.isna().sum())

Variables exogènes :
['heure_sin', 'heure_cos', 'jour_semaine_sin', 'jour_semaine_cos', 'mois_sin', 'mois_cos', 'weekend_utc', 'ferie', 'vacances']

Dimensions : (94224, 9)

Valeurs manquantes :
heure_sin           0
heure_cos           0
jour_semaine_sin    0
jour_semaine_cos    0
mois_sin            0
mois_cos            0
weekend_utc         0
ferie               0
vacances            0
dtype: int64


In [59]:
# ============================================================
# 53. PREMIER MODÈLE SARIMAX
# ============================================================

ordre = (1, 0, 1)
ordre_saisonnier = (1, 0, 1, 24)

print("Ordre SARIMAX :", ordre)
print("Ordre saisonnier :", ordre_saisonnier)
print("Saisonnalité : 24 heures")

Ordre SARIMAX : (1, 0, 1)
Ordre saisonnier : (1, 0, 1, 24)
Saisonnalité : 24 heures


In [60]:
# ============================================================
# 54. AJUSTEMENT DU SARIMAX
# ============================================================

modele_sarimax = SARIMAX(
    y_train,
    exog=X_train,
    order=ordre,
    seasonal_order=ordre_saisonnier,
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultat_sarimax = modele_sarimax.fit(
    disp=False
)

print(resultat_sarimax.summary())

e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


                                     SARIMAX Results                                      
Dep. Variable:                    consommation_mw   No. Observations:                43824
Model:             SARIMAX(1, 0, 1)x(1, 0, 1, 24)   Log Likelihood             -445429.276
Date:                            Thu, 08 Oct 2026   AIC                         890886.552
Time:                                    20:40:04   BIC                         891008.175
Sample:                                01-01-2019   HQIC                        890924.886
                                     - 12-31-2023                                         
Covariance Type:                              opg                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
heure_sin        -1254.0388    161.714     -7.755      0.000   -1570.992    -937.086
heure_cos        

In [ ]:
# ============================================================
# 55. PREMIÈRE PRÉVISION SARIMAX À 14H
# ============================================================

origine_paris = pd.Timestamp(
    "2024-01-15 14:00:00",
    tz="Europe/Paris"
)

# Conversion vers UTC pour retrouver l'observation dans df_model
origine_utc = origine_paris.tz_convert("UTC")

# Historique disponible à 14h J
y_historique = df_model.loc[
    :origine_utc,
    "consommation_mw"
]

# Les 24 heures futures de J+1
date_j1_paris = origine_paris.normalize() + pd.Timedelta(days=1)

heures_j1_paris = pd.date_range(
    start=date_j1_paris,
    periods=24,
    freq="h"
)

heures_j1_utc = heures_j1_paris.tz_convert("UTC")

# Variables calendaires connues à l'avance pour J+1
X_future = X_sarimax.loc[
    heures_j1_utc
]

print("Origine Paris :", origine_paris)
print("Origine UTC   :", origine_utc)

print("\nHistorique disponible jusqu'à :")
print(y_historique.index[-1])

print("\nNombre d'heures futures :", len(X_future))
print("Début :", X_future.index[0])
print("Fin   :", X_future.index[-1])

Origine Paris : 2024-01-15 14:00:00+01:00
Origine UTC   : 2024-01-15 13:00:00+00:00

Historique disponible jusqu'à :
2024-01-15 13:00:00+00:00

Nombre d'heures futures : 24
Début : 2024-01-15 23:00:00+00:00
Fin   : 2024-01-16 22:00:00+00:00


In [62]:
# ============================================================
# 56. PRÉVISION SARIMAX — 24 HEURES
# ============================================================

prevision_sarimax = resultat_sarimax.get_forecast(
    steps=24,
    exog=X_future
)

predictions = prevision_sarimax.predicted_mean

# Repasser les résultats sur l'index Paris pour faciliter la lecture
predictions_paris = pd.Series(
    predictions.values,
    index=heures_j1_paris,
    name="prediction_sarimax"
)

print("Prévisions SARIMAX :")
print(predictions_paris)

Prévisions SARIMAX :
2024-01-16 00:00:00+01:00   -21562.270988
2024-01-16 01:00:00+01:00    13928.300065
2024-01-16 02:00:00+01:00    15650.020790
2024-01-16 03:00:00+01:00    17769.887432
2024-01-16 04:00:00+01:00    19253.454758
2024-01-16 05:00:00+01:00    20286.920250
2024-01-16 06:00:00+01:00    21271.983285
2024-01-16 07:00:00+01:00    22396.938341
2024-01-16 08:00:00+01:00    23562.005025
2024-01-16 09:00:00+01:00    24532.422384
2024-01-16 10:00:00+01:00    25853.775874
2024-01-16 11:00:00+01:00    26924.019476
2024-01-16 12:00:00+01:00    28986.382445
2024-01-16 13:00:00+01:00    30887.006092
2024-01-16 14:00:00+01:00    31604.602893
2024-01-16 15:00:00+01:00    32319.951641
2024-01-16 16:00:00+01:00    31190.540801
2024-01-16 17:00:00+01:00    28286.201052
2024-01-16 18:00:00+01:00    26498.405846
2024-01-16 19:00:00+01:00    26352.461229
2024-01-16 20:00:00+01:00    26227.651513
2024-01-16 21:00:00+01:00    25217.316533
2024-01-16 22:00:00+01:00    22698.057399
2024-01-16 23

In [63]:
# ============================================================
# 57. VÉRIFICATION DE LA CONVERGENCE
# ============================================================

print("Convergence :", resultat_sarimax.mle_retvals.get("converged"))

print("\nNombre d'itérations :")
print(resultat_sarimax.mle_retvals.get("iterations"))

print("\nMessage de l'optimiseur :")
print(resultat_sarimax.mle_retvals.get("message"))

Convergence : True

Nombre d'itérations :
10

Message de l'optimiseur :
None


In [64]:
# ============================================================
# 58. SARIMA SANS VARIABLES EXOGÈNES
# ============================================================

modele_sarima = SARIMAX(
    y_train,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultat_sarima = modele_sarima.fit(
    disp=False
)

print("SARIMA ajusté avec succès.")
print("Convergence :",
      resultat_sarima.mle_retvals.get("converged"))

print("\nAIC :", resultat_sarima.aic)

e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


SARIMA ajusté avec succès.
Convergence : True

AIC : 679193.5698964081


In [65]:
# ============================================================
# 59. PRÉVISION SARIMA — 24 HEURES
# ============================================================

prevision_sarima = resultat_sarima.get_forecast(
    steps=24
)

predictions_sarima = prevision_sarima.predicted_mean

predictions_sarima_paris = pd.Series(
    predictions_sarima.values,
    index=heures_j1_paris,
    name="prediction_sarima"
)

print("Prévisions SARIMA :")
print(predictions_sarima_paris)

Prévisions SARIMA :
2024-01-16 00:00:00+01:00    52661.400419
2024-01-16 01:00:00+01:00    52135.923523
2024-01-16 02:00:00+01:00    49943.151357
2024-01-16 03:00:00+01:00    47985.989332
2024-01-16 04:00:00+01:00    47797.463612
2024-01-16 05:00:00+01:00    49112.869447
2024-01-16 06:00:00+01:00    50983.158618
2024-01-16 07:00:00+01:00    52809.212296
2024-01-16 08:00:00+01:00    54457.653647
2024-01-16 09:00:00+01:00    56160.649085
2024-01-16 10:00:00+01:00    57029.463731
2024-01-16 11:00:00+01:00    57944.316843
2024-01-16 12:00:00+01:00    56711.484969
2024-01-16 13:00:00+01:00    55185.591289
2024-01-16 14:00:00+01:00    54756.508494
2024-01-16 15:00:00+01:00    53614.623877
2024-01-16 16:00:00+01:00    54700.492394
2024-01-16 17:00:00+01:00    57951.593088
2024-01-16 18:00:00+01:00    59072.863453
2024-01-16 19:00:00+01:00    57138.834812
2024-01-16 20:00:00+01:00    54880.779332
2024-01-16 21:00:00+01:00    54031.635615
2024-01-16 22:00:00+01:00    55780.722002
2024-01-16 23:

In [66]:
# ============================================================
# 60. COMPARAISON SARIMA / RÉALITÉ
# ============================================================

reel_j1 = df_model.loc[
    heures_j1_utc,
    "consommation_mw"
]

comparaison_jour = pd.DataFrame({
    "reel_mw": reel_j1.values,
    "sarima_mw": predictions_sarima.values
}, index=heures_j1_paris)

comparaison_jour["erreur_mw"] = (
    comparaison_jour["sarima_mw"]
    - comparaison_jour["reel_mw"]
)

print(comparaison_jour)

                           reel_mw     sarima_mw     erreur_mw
2024-01-16 00:00:00+01:00  70752.5  52661.400419 -18091.099581
2024-01-16 01:00:00+01:00  68135.5  52135.923523 -15999.576477
2024-01-16 02:00:00+01:00  68245.5  49943.151357 -18302.348643
2024-01-16 03:00:00+01:00  66849.5  47985.989332 -18863.510668
2024-01-16 04:00:00+01:00  65473.5  47797.463612 -17676.036388
2024-01-16 05:00:00+01:00  66258.5  49112.869447 -17145.630553
2024-01-16 06:00:00+01:00  70274.5  50983.158618 -19291.341382
2024-01-16 07:00:00+01:00  76426.0  52809.212296 -23616.787704
2024-01-16 08:00:00+01:00  80628.5  54457.653647 -26170.846353
2024-01-16 09:00:00+01:00  81601.0  56160.649085 -25440.350915
2024-01-16 10:00:00+01:00  81412.5  57029.463731 -24383.036269
2024-01-16 11:00:00+01:00  80009.5  57944.316843 -22065.183157
2024-01-16 12:00:00+01:00  78564.0  56711.484969 -21852.515031
2024-01-16 13:00:00+01:00  75588.5  55185.591289 -20402.908711
2024-01-16 14:00:00+01:00  73944.5  54756.508494 -19187

In [67]:
# ============================================================
# 61. MÉTRIQUES SARIMA — JOURNÉE DU 16/01/2024
# ============================================================

mae_jour = mean_absolute_error(
    comparaison_jour["reel_mw"],
    comparaison_jour["sarima_mw"]
)

rmse_jour = np.sqrt(
    mean_squared_error(
        comparaison_jour["reel_mw"],
        comparaison_jour["sarima_mw"]
    )
)

print("=== SARIMA — 16/01/2024 ===")
print(f"MAE  : {mae_jour:.2f} MW")
print(f"RMSE : {rmse_jour:.2f} MW")

=== SARIMA — 16/01/2024 ===
MAE  : 19922.03 MW
RMSE : 20154.05 MW


In [68]:
# ============================================================
# 62. VARIABLES CALENDAIRES LOCALES ALIGNÉES SUR L'INDEX UTC
# ============================================================

df_model["heure_locale_sin"] = np.sin(
    2 * np.pi * df_model["heure"] / 24
)

df_model["heure_locale_cos"] = np.cos(
    2 * np.pi * df_model["heure"] / 24
)

df_model["jour_semaine_local_sin"] = np.sin(
    2 * np.pi * df_model["jour_semaine"] / 7
)

df_model["jour_semaine_local_cos"] = np.cos(
    2 * np.pi * df_model["jour_semaine"] / 7
)

df_model["mois_local_sin"] = np.sin(
    2 * np.pi * (df_model["mois"] - 1) / 12
)

df_model["mois_local_cos"] = np.cos(
    2 * np.pi * (df_model["mois"] - 1) / 12
)

print(
    df_model[
        [
            "timestamp_paris",
            "heure",
            "heure_locale_sin",
            "heure_locale_cos",
            "jour_semaine",
            "jour_semaine_local_sin",
            "jour_semaine_local_cos"
        ]
    ].head(10)
)

                                     timestamp_paris  heure  heure_locale_sin  \
timestamp_utc                                                                   
2016-01-01 00:00:00+00:00  2016-01-01 01:00:00+01:00      1          0.258819   
2016-01-01 01:00:00+00:00  2016-01-01 02:00:00+01:00      2          0.500000   
2016-01-01 02:00:00+00:00  2016-01-01 03:00:00+01:00      3          0.707107   
2016-01-01 03:00:00+00:00  2016-01-01 04:00:00+01:00      4          0.866025   
2016-01-01 04:00:00+00:00  2016-01-01 05:00:00+01:00      5          0.965926   
2016-01-01 05:00:00+00:00  2016-01-01 06:00:00+01:00      6          1.000000   
2016-01-01 06:00:00+00:00  2016-01-01 07:00:00+01:00      7          0.965926   
2016-01-01 07:00:00+00:00  2016-01-01 08:00:00+01:00      8          0.866025   
2016-01-01 08:00:00+00:00  2016-01-01 09:00:00+01:00      9          0.707107   
2016-01-01 09:00:00+00:00  2016-01-01 10:00:00+01:00     10          0.500000   

                           

In [69]:
# ============================================================
# 63. EXOGÈNES CALENDRIER LOCAL
# ============================================================

variables_exogenes_locales = [
    "heure_locale_sin",
    "heure_locale_cos",
    "jour_semaine_local_sin",
    "jour_semaine_local_cos",
    "mois_local_sin",
    "mois_local_cos",
    "ferie"
]

X_sarimax_local = df_model[
    variables_exogenes_locales
].copy()

X_sarimax_local["ferie"] = (
    X_sarimax_local["ferie"].astype(int)
)

print("Variables utilisées :")
print(X_sarimax_local.columns.tolist())

print("\nDimensions :", X_sarimax_local.shape)

print("\nValeurs manquantes :")
print(X_sarimax_local.isna().sum())

Variables utilisées :
['heure_locale_sin', 'heure_locale_cos', 'jour_semaine_local_sin', 'jour_semaine_local_cos', 'mois_local_sin', 'mois_local_cos', 'ferie']

Dimensions : (94224, 7)

Valeurs manquantes :
heure_locale_sin          0
heure_locale_cos          0
jour_semaine_local_sin    0
jour_semaine_local_cos    0
mois_local_sin            0
mois_local_cos            0
ferie                     0
dtype: int64


In [70]:
# ============================================================
# 64. SARIMAX — CALENDRIER LOCAL
# ============================================================

modele_sarimax_local = SARIMAX(
    y_train,
    exog=X_sarimax_local.loc[y_train.index],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultat_sarimax_local = modele_sarimax_local.fit(
    disp=False
)

print("SARIMAX calendrier local ajusté avec succès.")

print(
    "Convergence :",
    resultat_sarimax_local.mle_retvals.get("converged")
)

print(
    "AIC :",
    resultat_sarimax_local.aic
)

e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


SARIMAX calendrier local ajusté avec succès.
Convergence : True
AIC : 812428.1311563269


In [71]:
# ============================================================
# 65. EXOGÈNES FUTURES — 16/01/2024
# ============================================================

X_future_local = X_sarimax_local.loc[
    heures_j1_utc
].copy()

print("Dimensions :", X_future_local.shape)

print("\nDébut :", X_future_local.index[0])
print("Fin   :", X_future_local.index[-1])

print("\nValeurs manquantes :")
print(X_future_local.isna().sum())

Dimensions : (24, 7)

Début : 2024-01-15 23:00:00+00:00
Fin   : 2024-01-16 22:00:00+00:00

Valeurs manquantes :
heure_locale_sin          0
heure_locale_cos          0
jour_semaine_local_sin    0
jour_semaine_local_cos    0
mois_local_sin            0
mois_local_cos            0
ferie                     0
dtype: int64


In [72]:
# ============================================================
# 66. PRÉVISION SARIMAX — CALENDRIER LOCAL
# ============================================================

prevision_sarimax_local = (
    resultat_sarimax_local.get_forecast(
        steps=24,
        exog=X_future_local
    )
)

predictions_sarimax_local = (
    prevision_sarimax_local.predicted_mean
)

predictions_sarimax_local_paris = pd.Series(
    predictions_sarimax_local.values,
    index=heures_j1_paris,
    name="prediction_sarimax_local"
)

print("Prévisions SARIMAX — calendrier local :")
print(predictions_sarimax_local_paris)

Prévisions SARIMAX — calendrier local :
2024-01-16 00:00:00+01:00     8214.527539
2024-01-16 01:00:00+01:00     8960.125533
2024-01-16 02:00:00+01:00     8106.071095
2024-01-16 03:00:00+01:00     7738.328794
2024-01-16 04:00:00+01:00     9385.447900
2024-01-16 05:00:00+01:00    13223.635458
2024-01-16 06:00:00+01:00    18066.825911
2024-01-16 07:00:00+01:00    21692.253492
2024-01-16 08:00:00+01:00    24174.791822
2024-01-16 09:00:00+01:00    26177.785496
2024-01-16 10:00:00+01:00    27673.026058
2024-01-16 11:00:00+01:00    29267.651480
2024-01-16 12:00:00+01:00    28963.424619
2024-01-16 13:00:00+01:00    28849.355243
2024-01-16 14:00:00+01:00    29523.659397
2024-01-16 15:00:00+01:00    29736.821403
2024-01-16 16:00:00+01:00    32163.275496
2024-01-16 17:00:00+01:00    36518.318869
2024-01-16 18:00:00+01:00    39009.877034
2024-01-16 19:00:00+01:00    37773.696251
2024-01-16 20:00:00+01:00    35824.591114
2024-01-16 21:00:00+01:00    35163.735920
2024-01-16 22:00:00+01:00    36824.4

In [73]:
# ============================================================
# 67. CONSOMMATION RÉCENTE DISPONIBLE À 14H
# ============================================================

print("Dernières observations disponibles à 14h :")

print(
    df_model.loc[
        origine_utc - pd.Timedelta(hours=24):
        origine_utc,
        ["consommation_mw"]
    ].tail(25)
)

Dernières observations disponibles à 14h :
                           consommation_mw
timestamp_utc                             
2024-01-14 13:00:00+00:00          70646.0
2024-01-14 14:00:00+00:00          69511.0
2024-01-14 15:00:00+00:00          68201.0
2024-01-14 16:00:00+00:00          68270.5
2024-01-14 17:00:00+00:00          72425.5
2024-01-14 18:00:00+00:00          75830.0
2024-01-14 19:00:00+00:00          74666.5
2024-01-14 20:00:00+00:00          71699.0
2024-01-14 21:00:00+00:00          69597.5
2024-01-14 22:00:00+00:00          70307.5
2024-01-14 23:00:00+00:00          68736.0
2024-01-15 00:00:00+00:00          65963.5
2024-01-15 01:00:00+00:00          65633.5
2024-01-15 02:00:00+00:00          63940.0
2024-01-15 03:00:00+00:00          62250.5
2024-01-15 04:00:00+00:00          62957.5
2024-01-15 05:00:00+00:00          66961.0
2024-01-15 06:00:00+00:00          72906.5
2024-01-15 07:00:00+00:00          76962.0
2024-01-15 08:00:00+00:00          78201.5
2024-01-15 

In [74]:
# ============================================================
# 67. LAG J-7 : INFORMATION DISPONIBLE À 14H J
# ============================================================

df_model["consommation_lag_168"] = (
    df_model["consommation_mw"].shift(168)
)

print(
    df_model[
        ["consommation_mw", "consommation_lag_168"]
    ].loc["2024-01-15 00:00":"2024-01-15 13:00"]
)

                           consommation_mw  consommation_lag_168
timestamp_utc                                                   
2024-01-15 00:00:00+00:00          65963.5               60892.0
2024-01-15 01:00:00+00:00          65633.5               60949.5
2024-01-15 02:00:00+00:00          63940.0               59536.5
2024-01-15 03:00:00+00:00          62250.5               58248.0
2024-01-15 04:00:00+00:00          62957.5               59446.0
2024-01-15 05:00:00+00:00          66961.0               63995.0
2024-01-15 06:00:00+00:00          72906.5               70512.0
2024-01-15 07:00:00+00:00          76962.0               75060.0
2024-01-15 08:00:00+00:00          78201.5               76991.0
2024-01-15 09:00:00+00:00          78519.5               77991.5
2024-01-15 10:00:00+00:00          77923.0               78483.5
2024-01-15 11:00:00+00:00          77337.0               79079.0
2024-01-15 12:00:00+00:00          74864.0               77654.5
2024-01-15 13:00:00+00:00

In [75]:
# ============================================================
# 68. SARIMAX : CALENDRIER LOCAL + CONSOMMATION J-7
# ============================================================

variables_exog_lag168 = [
    "heure_locale_sin",
    "heure_locale_cos",
    "jour_semaine_local_sin",
    "jour_semaine_local_cos",
    "mois_local_sin",
    "mois_local_cos",
    "ferie",
    "consommation_lag_168"
]

X_sarimax_lag168 = df_model[variables_exog_lag168].copy()

print("Variables exogènes :")
print(variables_exog_lag168)

print("\nShape :", X_sarimax_lag168.shape)
print("\nValeurs manquantes :")
print(X_sarimax_lag168.isna().sum())

Variables exogènes :
['heure_locale_sin', 'heure_locale_cos', 'jour_semaine_local_sin', 'jour_semaine_local_cos', 'mois_local_sin', 'mois_local_cos', 'ferie', 'consommation_lag_168']

Shape : (94224, 8)

Valeurs manquantes :
heure_locale_sin            0
heure_locale_cos            0
jour_semaine_local_sin      0
jour_semaine_local_cos      0
mois_local_sin              0
mois_local_cos              0
ferie                       0
consommation_lag_168      168
dtype: int64


In [76]:
# ============================================================
# 69. DONNÉES D'ENTRAÎNEMENT — SARIMAX + LAG J-7
# ============================================================

train_mask = (
    (df_model.index >= "2019-01-01") &
    (df_model.index <= "2023-12-31 23:00:00+00:00")
)

y_train_lag168 = df_model.loc[
    train_mask, "consommation_mw"
]

X_train_lag168 = X_sarimax_lag168.loc[
    train_mask
]

# Retirer les lignes avec lag_168 manquant
masque_valide = X_train_lag168["consommation_lag_168"].notna()

y_train_lag168 = y_train_lag168.loc[masque_valide]
X_train_lag168 = X_train_lag168.loc[masque_valide]

print("y_train :", y_train_lag168.shape)
print("X_train :", X_train_lag168.shape)

print("\nPériode :")
print(y_train_lag168.index.min())
print(y_train_lag168.index.max())

print("\nValeurs manquantes :")
print(X_train_lag168.isna().sum().sum())

y_train : (43824,)
X_train : (43824, 8)

Période :
2019-01-01 00:00:00+00:00
2023-12-31 23:00:00+00:00

Valeurs manquantes :
0


In [77]:
# ============================================================
# 70. VÉRIFICATION DES TYPES
# ============================================================

print(X_train_lag168.dtypes)

print("\nTypes de y :")
print(y_train_lag168.dtype)

print("\nColonnes de type object :")
print(
    X_train_lag168.select_dtypes(include=["object"]).columns.tolist()
)

heure_locale_sin          float64
heure_locale_cos          float64
jour_semaine_local_sin    float64
jour_semaine_local_cos    float64
mois_local_sin            float64
mois_local_cos            float64
ferie                        bool
consommation_lag_168      float64
dtype: object

Types de y :
float64

Colonnes de type object :
[]


In [78]:

X_train_lag168 = X_train_lag168.astype(float)

print(X_train_lag168.dtypes)
print("\nType général :", X_train_lag168.dtypes.unique())

heure_locale_sin          float64
heure_locale_cos          float64
jour_semaine_local_sin    float64
jour_semaine_local_cos    float64
mois_local_sin            float64
mois_local_cos            float64
ferie                     float64
consommation_lag_168      float64
dtype: object

Type général : [dtype('float64')]


In [ ]:
# ============================================================
# 72. ENTRAÎNEMENT SARIMAX + LAG J-7
# ============================================================

modele_sarimax_lag168 = SARIMAX(
    y_train_lag168,
    exog=X_train_lag168,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultat_sarimax_lag168 = modele_sarimax_lag168.fit(
    disp=False
)

print(resultat_sarimax_lag168.summary())
print(
    "\nConvergence :",
    resultat_sarimax_lag168.mle_retvals["converged"]
)

e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
e:\Projects\Studies\ElectricityConsumption\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


In [ ]:
# ============================================================
# 73. VARIABLES EXOGÈNES FUTURES — J+1
# ============================================================

X_future_lag168 = X_sarimax_lag168.loc[
    heures_j1_utc,
    variables_exog_lag168
].copy()

# Même conversion numérique que pour l'entraînement
X_future_lag168 = X_future_lag168.astype(float)

print("Shape :", X_future_lag168.shape)
print("\nValeurs manquantes :")
print(X_future_lag168.isna().sum())

print("\nAperçu :")
print(X_future_lag168.head())

Shape : (24, 8)

Valeurs manquantes :
heure_locale_sin          0
heure_locale_cos          0
jour_semaine_local_sin    0
jour_semaine_local_cos    0
mois_local_sin            0
mois_local_cos            0
ferie                     0
consommation_lag_168      0
dtype: int64

Aperçu :
                           heure_locale_sin  heure_locale_cos  \
2024-01-15 23:00:00+00:00          0.000000          1.000000   
2024-01-16 00:00:00+00:00          0.258819          0.965926   
2024-01-16 01:00:00+00:00          0.500000          0.866025   
2024-01-16 02:00:00+00:00          0.707107          0.707107   
2024-01-16 03:00:00+00:00          0.866025          0.500000   

                           jour_semaine_local_sin  jour_semaine_local_cos  \
2024-01-15 23:00:00+00:00                0.781831                 0.62349   
2024-01-16 00:00:00+00:00                0.781831                 0.62349   
2024-01-16 01:00:00+00:00                0.781831                 0.62349   
2024-01-16 02:00

In [ ]:
# ============================================================
# 74. PRÉVISION SARIMAX + LAG J-7
# ============================================================

prevision_lag168 = resultat_sarimax_lag168.get_forecast(
    steps=24,
    exog=X_future_lag168
)

predictions_lag168 = prevision_lag168.predicted_mean

predictions_lag168_paris = pd.Series(
    predictions_lag168.values,
    index=heures_j1_paris,
    name="prediction_sarimax_lag168"
)

print("Prévisions SARIMAX + J-7 :")
print(predictions_lag168_paris)


Prévisions SARIMAX + J-7 :
2024-01-16 00:00:00+01:00    77659.398399
2024-01-16 01:00:00+01:00    75195.535724
2024-01-16 02:00:00+01:00    74909.746654
2024-01-16 03:00:00+01:00    72983.466248
2024-01-16 04:00:00+01:00    71341.192691
2024-01-16 05:00:00+01:00    71653.962275
2024-01-16 06:00:00+01:00    74352.265070
2024-01-16 07:00:00+01:00    78848.450878
2024-01-16 08:00:00+01:00    82449.099155
2024-01-16 09:00:00+01:00    83973.848470
2024-01-16 10:00:00+01:00    84703.219537
2024-01-16 11:00:00+01:00    84893.908524
2024-01-16 12:00:00+01:00    84672.971571
2024-01-16 13:00:00+01:00    82693.491096
2024-01-16 14:00:00+01:00    81462.689035
2024-01-16 15:00:00+01:00    80666.671921
2024-01-16 16:00:00+01:00    79861.611814
2024-01-16 17:00:00+01:00    80942.794477
2024-01-16 18:00:00+01:00    84192.693606
2024-01-16 19:00:00+01:00    86057.516380
2024-01-16 20:00:00+01:00    83731.448379
2024-01-16 21:00:00+01:00    80183.177015
2024-01-16 22:00:00+01:00    78206.601036
2024-01

In [ ]:
# ============================================================
# 75. ÉVALUATION SUR LE 16/01/2024
# ============================================================

actual_2024_01_16 = df_model.loc[
    heures_j1_utc,
    "consommation_mw"
]

y_true = actual_2024_01_16.values
y_pred = predictions_lag168.values

mae_lag168 = mean_absolute_error(y_true, y_pred)
rmse_lag168 = np.sqrt(
    mean_squared_error(y_true, y_pred)
)

print("Évaluation SARIMAX + J-7")
print("------------------------")
print(f"MAE  : {mae_lag168:.2f} MW")
print(f"RMSE : {rmse_lag168:.2f} MW")


Évaluation SARIMAX + J-7
------------------------
MAE  : 5825.21 MW
RMSE : 6083.82 MW


In [ ]:
# ============================================================
# 76. COMPARAISON SARIMAX + J-7 / BENCHMARKS
# ============================================================

# Benchmark J-7 pour cette journée
benchmark_j7_2024 = benchmark_14h[
    benchmark_14h.index.isin(heures_j1_paris)
].copy()

# Réalignement sur les heures locales
benchmark_j7_2024.index = heures_j1_paris

# Valeurs réelles
actual_paris = pd.Series(
    actual_2024_01_16.values,
    index=heures_j1_paris
)

# Comparaison
comparaison_2024_01_16 = pd.DataFrame({
    "reel": actual_paris,
    "sarimax_j7": predictions_lag168_paris,
    "benchmark_j7": benchmark_j7_2024
})

print(comparaison_2024_01_16)

ValueError: Length mismatch: Expected axis has 0 elements, new values have 24 elements